# Game-Grouped Validation

A route-random split can place routes from the same game in train and test. This experiment assigns whole games to one partition, stratifying the random assignment within each week so every week appears in train, validation, and test.

This tests whether performance changes when the model is evaluated on entirely unseen games from the same season.

In [1]:
from pathlib import Path
import sys
from collections import Counter

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from separation_over_expected.reports import read_csv_rows

GAME_DIR = PROJECT_ROOT / "data" / "processed" / "game_split" / "position_baselines"
ROUTE_DIR = PROJECT_ROOT / "data" / "processed" / "random_split" / "position_baselines"
rows = read_csv_rows(GAME_DIR / "route_level_baseline_predictions_all.csv")
len(rows)

35032

## Confirm Game Separation

Game assignments are made within week, then shared by all route rows from that game. The counts below verify that no game appears in multiple partitions and that the split covers every week.

In [2]:
rows_by_split = {
    split: [row for row in rows if row["split"] == split]
    for split in ("train", "validation", "test")
}
games_by_split = {
    split: {row["gameId"] for row in split_rows}
    for split, split_rows in rows_by_split.items()
}
overlaps = {
    f"{left}/{right}": len(games_by_split[left] & games_by_split[right])
    for left, right in (("train", "validation"), ("train", "test"), ("validation", "test"))
}
split_summary = {
    split: {
        "routes": len(split_rows),
        "games": len(games_by_split[split]),
        "routes_by_week": dict(sorted(Counter(row["week"] for row in split_rows).items())),
    }
    for split, split_rows in rows_by_split.items()
}
{"splits": split_summary, "game_overlap": overlaps}

{'splits': {'train': {'routes': 24463, 'games': 85, 'routes_by_week': {'1': 3512, '2': 2976, '3': 3132, '4': 3207, '5': 3140, '6': 2924, '7': 2494, '8': 3078}}, 'validation': {'routes': 4497, 'games': 16, 'routes_by_week': {'1': 553, '2': 612, '3': 578, '4': 577, '5': 596, '6': 536, '7': 582, '8': 463}}, 'test': {'routes': 6072, 'games': 21, 'routes_by_week': {'1': 862, '2': 731, '3': 947, '4': 690, '5': 809, '6': 702, '7': 620, '8': 711}}}, 'game_overlap': {'train/validation': 0, 'train/test': 0, 'validation/test': 0}}

## Test-Set Comparison

Compare every baseline model under route-random and game-grouped splits. Both use seed 42 and all eight weeks; the game split is stratified by week. Because grouping changes the test sample, score differences are a sensitivity check, not a pure estimate of within-game leakage.

In [3]:
positions = [("All", "all"), ("WR", "wr"), ("TE", "te"), ("RB", "rb")]
comparison = []
for position, suffix in positions:
    metrics_by_strategy = {}
    for strategy, directory in (("Route random", ROUTE_DIR), ("Game grouped", GAME_DIR)):
        metric_rows = read_csv_rows(directory / f"baseline_metrics_{suffix}.csv")
        metrics_by_strategy[strategy] = {
            row["model"]: row for row in metric_rows if row["split"] == "test"
        }
    for model in ("global_mean", "smoothed_group_mean", "ridge_context"):
        route = metrics_by_strategy["Route random"][model]
        game = metrics_by_strategy["Game grouped"][model]
        comparison.append({
            "position": position,
            "model": model,
            "route_test_n": route["n"],
            "game_test_n": game["n"],
            "route_r2": route["r2"],
            "game_r2": game["r2"],
            "route_rmse": route["rmse"],
            "game_rmse": game["rmse"],
            "route_mae": route["mae"],
            "game_mae": game["mae"],
        })
comparison

[{'position': 'All', 'model': 'global_mean', 'route_test_n': '5256', 'game_test_n': '6072', 'route_r2': '-0.000', 'game_r2': '-0.001', 'route_rmse': '2.953', 'game_rmse': '2.945', 'route_mae': '2.236', 'game_mae': '2.236'}, {'position': 'All', 'model': 'smoothed_group_mean', 'route_test_n': '5256', 'game_test_n': '6072', 'route_r2': '0.050', 'game_r2': '0.051', 'route_rmse': '2.878', 'game_rmse': '2.868', 'route_mae': '2.187', 'game_mae': '2.180'}, {'position': 'All', 'model': 'ridge_context', 'route_test_n': '5256', 'game_test_n': '6072', 'route_r2': '0.359', 'game_r2': '0.340', 'route_rmse': '2.363', 'game_rmse': '2.392', 'route_mae': '1.711', 'game_mae': '1.746'}, {'position': 'WR', 'model': 'global_mean', 'route_test_n': '3082', 'game_test_n': '3527', 'route_r2': '-0.000', 'game_r2': '-0.001', 'route_rmse': '2.717', 'game_rmse': '2.726', 'route_mae': '2.078', 'game_mae': '2.090'}, {'position': 'WR', 'model': 'smoothed_group_mean', 'route_test_n': '3082', 'game_test_n': '3527', 'rou

## Ridge Context Results

The grouped split modestly lowers test R2 for the pooled model, WRs, and TEs. RB R2 is nearly unchanged. WR RMSE increases by about 0.07 yards, while TE and RB RMSE are nearly unchanged. This pattern suggests some optimism in the route-level split, especially for WRs, but it does not isolate the cause because the held-out games differ.

In [4]:
[row for row in comparison if row["model"] == "ridge_context"]

[{'position': 'All', 'model': 'ridge_context', 'route_test_n': '5256', 'game_test_n': '6072', 'route_r2': '0.359', 'game_r2': '0.340', 'route_rmse': '2.363', 'game_rmse': '2.392', 'route_mae': '1.711', 'game_mae': '1.746'}, {'position': 'WR', 'model': 'ridge_context', 'route_test_n': '3082', 'game_test_n': '3527', 'route_r2': '0.514', 'game_r2': '0.479', 'route_rmse': '1.895', 'game_rmse': '1.967', 'route_mae': '1.381', 'game_mae': '1.432'}, {'position': 'TE', 'model': 'ridge_context', 'route_test_n': '1173', 'game_test_n': '1390', 'route_r2': '0.288', 'game_r2': '0.266', 'route_rmse': '2.373', 'game_rmse': '2.369', 'route_mae': '1.704', 'game_mae': '1.724'}, {'position': 'RB', 'model': 'ridge_context', 'route_test_n': '939', 'game_test_n': '1070', 'route_r2': '0.235', 'game_r2': '0.237', 'route_rmse': '3.241', 'game_rmse': '3.243', 'route_mae': '2.582', 'game_mae': '2.546'}]

## Interpretation

Treat route-random performance as the estimate for held-out routes from the season-wide mix, and game-grouped performance as a stricter check on shared game context. Neither split tests generalization to a different NFL season.